# Day 9 目标：把 RAG 接入 LangGraph

今天的核心目标是：

把 Day 8 的 retrieve_documents() 接进阶段一的任务节点里，让 paper_node / experiment_node / dataset_node 不再只返回 mock 文本，而是能检索真实资料并输出 Sources。

完成后，你的流程会从：

用户输入
↓
classify_task
↓
route_task
↓
mock 节点回答
↓
final_answer

升级为：

用户输入
↓
classify_task 得到 task_type
↓
route_task 进入对应节点
↓
节点调用 RAG Retriever
↓
检索相关 Document chunks
↓
生成回答摘要
↓
final_answer 输出回答 + Sources

# 一、Day 9 要改哪些文件？

今天主要修改：

F:\ResearchAgent
├── src
│   └── research_agent
│       └── graph
│           ├── state.py
│           └── nodes.py
│
├── run_cli.py
└── scripts
    └── test_agentic_rag.py

今天不改或少改：

workflow.py
router.py
llm_classifier.py
indexer.py
retriever.py

因为 Day 8 的 retriever 已经做好了，今天只是调用它。

# 二、先确认 Day 8 索引存在

Day 9 依赖本地 Chroma 向量库：

F:\ResearchAgent\storage\chroma_db

如果你不确定，先运行：

cd F:\ResearchAgent
.\.conda\python.exe scripts\build_index.py

然后测试：

.\.conda\python.exe scripts\test_retriever_routing.py

确认能看到：

All source_type match expected: True

# 三、修改 state.py

路径：

F:\ResearchAgent\src\research_agent\graph\state.py

把它改成：

from typing import TypedDict, List, Dict


class AgentState(TypedDict):
    query: str
    task_type: str
    result: str
    final_answer: str

    classifier_source: str
    route_reason: str

    # Day 9 新增：RAG 检索结果
    retrieved_docs: List[Dict]
    sources: List[Dict]
新增字段解释
| 字段               | 作用                        |
| ---------------- | ------------------------- |
| `retrieved_docs` | 保存检索到的 chunk 内容和 metadata |
| `sources`        | 保存去重后的来源文件列表              |
| `result`         | 当前节点基于检索结果生成的回答           |
| `final_answer`   | 最终展示文本                    |


# 四、修改 nodes.py

路径：

F:\ResearchAgent\src\research_agent\graph\nodes.py
1. 在文件顶部新增导入

你原本可能已经有：

from .state import AgentState
from .llm_classifier import classify_with_llm, get_llm_classifier_enabled

现在加上：

from research_agent.rag.retriever import (
    retrieve_documents,
    document_to_dict,
    extract_sources_from_docs,
    format_retrieved_docs,
)

如果你的项目暂时还是用 from src... 风格运行，也可以写：

from src.research_agent.rag.retriever import (
    retrieve_documents,
    document_to_dict,
    extract_sources_from_docs,
    format_retrieved_docs,
)

但如果你的 scripts 里已经使用 SRC_DIR = PROJECT_ROOT / "src"，更推荐统一为：

from research_agent.rag.retriever import ...

# 五、添加一个通用 RAG 辅助函数

在 nodes.py 中加入这个函数，建议放在分类函数后面、各任务节点前面：

def run_rag_for_task(state: AgentState, task_type: str, top_k: int = 3) -> dict:
    """
    根据 query 和 task_type 调用 Retriever，
    返回 retrieved_docs、sources 和格式化后的 retrieved_context。
    """
    query = state["query"]

    docs = retrieve_documents(
        query=query,
        task_type=task_type,
        top_k=top_k,
        use_filter=True,
    )

    retrieved_docs = [document_to_dict(doc) for doc in docs]
    sources = extract_sources_from_docs(docs)
    retrieved_context = format_retrieved_docs(docs, max_chars_per_doc=500)

    return {
        "retrieved_docs": retrieved_docs,
        "sources": sources,
        "retrieved_context": retrieved_context,
    }

注意：retrieved_context 不一定要写进 AgentState，它只是当前函数内部辅助生成回答用。

# 六、把三个核心节点改成 RAG 版本

今天优先改这三个：

paper_node
experiment_node
dataset_node

code_node / report_node / general_node 暂时可以继续 mock。

1. 修改 paper_node
def paper_node(state: AgentState) -> dict:
    rag_result = run_rag_for_task(
        state=state,
        task_type="paper_question",
        top_k=3,
    )

    result = f"""
这是论文问答任务。系统已从论文笔记资料库中检索到相关内容。

基于检索资料的初步回答：
{build_simple_answer_from_context(state["query"], rag_result["retrieved_context"])}
""".strip()

    return {
        "retrieved_docs": rag_result["retrieved_docs"],
        "sources": rag_result["sources"],
        "result": result,
    }
2. 修改 experiment_node
def experiment_node(state: AgentState) -> dict:
    rag_result = run_rag_for_task(
        state=state,
        task_type="experiment_analysis",
        top_k=3,
    )

    result = f"""
这是实验分析任务。系统已从实验说明资料库中检索到相关内容。

基于检索资料的初步回答：
{build_simple_answer_from_context(state["query"], rag_result["retrieved_context"])}
""".strip()

    return {
        "retrieved_docs": rag_result["retrieved_docs"],
        "sources": rag_result["sources"],
        "result": result,
    }
3. 修改 dataset_node
def dataset_node(state: AgentState) -> dict:
    rag_result = run_rag_for_task(
        state=state,
        task_type="dataset_recommendation",
        top_k=3,
    )

    result = f"""
这是数据集推荐 / 数据集说明任务。系统已从数据集资料库中检索到相关内容。

基于检索资料的初步回答：
{build_simple_answer_from_context(state["query"], rag_result["retrieved_context"])}
""".strip()

    return {
        "retrieved_docs": rag_result["retrieved_docs"],
        "sources": rag_result["sources"],
        "result": result,
    }

# 七、添加一个最小回答生成函数

今天先不强制接 LLM 生成最终回答，先做一个基于检索内容的简单回答器。

在 nodes.py 里加入：

def build_simple_answer_from_context(query: str, retrieved_context: str) -> str:
    """
    v0.2 最小回答生成器。

    当前版本不调用 LLM，只把检索上下文整理成可读回答。
    后续可以替换为 LLM-based answer generator。
    """
    if not retrieved_context or retrieved_context == "未检索到相关资料。":
        return "暂未在本地科研资料库中检索到足够相关的资料。"

    return f"""
用户问题：{query}

检索到的相关资料摘要如下：
{retrieved_context}

说明：当前 v0.2 版本暂时使用检索内容摘要作为回答，后续可以接入 LLM 对 retrieved_docs 进行综合生成。
""".strip()

这不是最终形态，但非常适合 Day 9 验收：

能检索
能显示内容
能显示来源
能跑通 LangGraph

不要今天就把回答生成做复杂。

# 八、保留其他节点

report_node 可以暂时这样：

def report_node(state: AgentState) -> dict:
    return {
        "retrieved_docs": [],
        "sources": [],
        "result": "这是汇报生成任务，后续会接入 Report Writer。"
    }

code_node：

def code_node(state: AgentState) -> dict:
    return {
        "retrieved_docs": [],
        "sources": [],
        "result": "这是代码辅助任务，后续会接入代码解释与修改工具。"
    }

general_node：

def general_node(state: AgentState) -> dict:
    return {
        "retrieved_docs": [],
        "sources": [],
        "result": "这是通用科研助手任务。"
    }

# 九、修改 final_answer_node

把最终回答改成带 Sources 的版本：

def format_sources(sources: list) -> str:
    """
    格式化 Sources。
    """
    if not sources:
        return "无"

    lines = []

    for i, source in enumerate(sources, start=1):
        path = source.get("path", "unknown")
        source_type = source.get("source_type", "unknown")
        title = source.get("title", "")
        dataset = source.get("dataset", "")
        run_tag = source.get("run_tag", "")

        extra_parts = []

        if title:
            extra_parts.append(f"title={title}")

        if dataset:
            extra_parts.append(f"dataset={dataset}")

        if run_tag:
            extra_parts.append(f"run_tag={run_tag}")

        extra = f" ({', '.join(extra_parts)})" if extra_parts else ""

        lines.append(f"{i}. [{source_type}] {path}{extra}")

    return "\n".join(lines)


def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
分类来源：{state["classifier_source"]}
分类原因：{state["route_reason"]}

回答：
{state["result"]}

Sources:
{format_sources(state.get("sources", []))}
""".strip()

    return {
        "final_answer": final_answer
    }

# 十、修改 run_cli.py

因为 AgentState 新增了：

retrieved_docs
sources

所以初始 State 要补上。

路径：

F:\ResearchAgent\run_cli.py

修改 create_initial_state()：

def create_initial_state(query: str) -> dict:
    return {
        "query": query,
        "task_type": "",
        "result": "",
        "final_answer": "",
        "classifier_source": "",
        "route_reason": "",
        "retrieved_docs": [],
        "sources": [],
    }

其他地方不用大改。

# 十一、新建测试脚本 test_agentic_rag.py

路径：

F:\ResearchAgent\scripts\test_agentic_rag.py

复制：

from pathlib import Path
import sys


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(PROJECT_ROOT))


from run_cli import create_initial_state
from research_agent.graph.workflow import build_graph


TEST_QUERIES = [
    "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
    "请帮我解释 coco_val_n300_g1 这个实验的目的",
    "Guardrail-Agnostic 这篇论文适合放在哪类 related work？",
    "GQA 对场景图和关系分析有什么价值？",
    "ModuleNotFoundError: No module named langgraph 怎么解决",
]


def main():
    graph = build_graph()

    for query in TEST_QUERIES:
        print("=" * 100)
        print("用户输入：", query)
        print("=" * 100)

        result = graph.invoke(create_initial_state(query))

        print(result["final_answer"])
        print("\nretrieved_docs count:", len(result.get("retrieved_docs", [])))
        print("sources count:", len(result.get("sources", [])))
        print()


if __name__ == "__main__":
    main()

# 十二、运行 Day 9 测试

先构建索引：

cd F:\ResearchAgent
.\.conda\python.exe scripts\build_index.py

再运行：

.\.conda\python.exe scripts\test_agentic_rag.py

如果没有问题，你应该看到：

用户输入：OpenImages-MIAP 的性别标注是图像级还是 bbox 级？

任务类型：dataset_recommendation
分类来源：rule / llm
分类原因：...

回答：
这是数据集推荐 / 数据集说明任务。系统已从数据集资料库中检索到相关内容。

基于检索资料的初步回答：
用户问题：OpenImages-MIAP 的性别标注是图像级还是 bbox 级？

检索到的相关资料摘要如下：
[1] source_type=dataset_doc | path=data/datasets/openimages_miap.md ...

Sources:
1. [dataset_doc] data/datasets/openimages_miap.md ...

# 十三、再运行 CLI Demo
.\.conda\python.exe run_cli.py

测试输入：

OpenImages-MIAP 的性别标注是图像级还是 bbox 级？

再测试：

请帮我解释 coco_val_n300_g1 这个实验的目的

再测试：

Guardrail-Agnostic 这篇论文适合放在哪类 related work？

# 十四、今天常见错误
1. Chroma index not found

说明还没构建索引。

运行：

.\.conda\python.exe scripts\build_index.py
2. KeyError: 'retrieved_docs' 或 KeyError: 'sources'

说明 run_cli.py 的初始 State 没加：

"retrieved_docs": [],
"sources": [],

补上即可。

3. ModuleNotFoundError: No module named 'research_agent'

说明脚本没有把 src 加入路径。

在脚本前面加：

from pathlib import Path
import sys

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))
4. 分类错了，导致没有查对应资料

比如用户问：

OpenImages-MIAP 的性别标注是图像级还是 bbox 级？

如果分类成 general，它就不会进入 dataset_node。

解决方式：

在 classify_task_by_rule() 中增强规则，比如加入：

elif (
    "数据集" in query
    or "dataset" in query.lower()
    or "OpenImages" in query
    or "MIAP" in query
    or "FairFace" in query
    or "GQA" in query
    or "COCO" in query
):
    task_type = "dataset_recommendation"

但注意：COCO 有时也可能是实验问题，比如 coco_val_n300_g1。所以更精确的顺序应该是：

先判断 run_tag / 实验关键词
再判断数据集关键词

# 十五、Day 9 验收标准

今天完成后，你应该做到：

1. AgentState 新增 retrieved_docs 和 sources
2. paper_node 能调用 RAG 检索 paper_note
3. experiment_node 能调用 RAG 检索 experiment_doc
4. dataset_node 能调用 RAG 检索 dataset_doc
5. final_answer 能输出 Sources
6. run_cli.py 能跑通 v0.2 Agentic RAG demo
7. test_agentic_rag.py 能一键测试多个问题
8. 对 dataset / experiment / paper 问题，retrieved_docs count > 0
9. 对 dataset / experiment / paper 问题，sources count > 0

# Day 9 一句话总结

Day 5–8 你是在搭 RAG 的底层能力：

写资料 → 加载 Document → 建向量库 → 按任务类型检索

Day 9 是把它接进 Agent：

LangGraph 节点 → 调用 Retriever → 返回回答和 Sources

完成后，你的 ResearchAgent 就从：

v0.1：LangGraph mock demo

升级成：

v0.2：Agentic RAG demo

这就是阶段二的阶段性成果。

# 关于state目前流转的解释

可以。我们就以你 Day 9 的测试脚本 `test_agentic_rag.py` 为例，把 **LangGraph 从 `graph.invoke()` 开始，到最终输出 `final_answer` 的全过程**讲清楚。

---

# 1. 先看测试脚本做了什么

你的测试脚本核心大概是：

```python
from pathlib import Path
import sys


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(PROJECT_ROOT))


from run_cli import create_initial_state
from research_agent.graph.workflow import build_graph


TEST_QUERIES = [
    "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
    "请帮我解释 coco_val_n300_g1 这个实验的目的",
    "Guardrail-Agnostic 这篇论文适合放在哪类 related work？",
    "GQA 对场景图和关系分析有什么价值？",
    "ModuleNotFoundError: No module named langgraph 怎么解决",
]


def main():
    graph = build_graph()

    for query in TEST_QUERIES:
        print("=" * 100)
        print("用户输入：", query)
        print("=" * 100)

        result = graph.invoke(create_initial_state(query))

        print(result["final_answer"])
        print("\nretrieved_docs count:", len(result.get("retrieved_docs", [])))
        print("sources count:", len(result.get("sources", [])))
        print()


if __name__ == "__main__":
    main()
```

这个脚本可以分成三部分：

```text
1. 设置导入路径
2. 构建 LangGraph
3. 循环测试多个 query
```

其中真正触发 Agent 工作流的是这一句：

```python
result = graph.invoke(create_initial_state(query))
```

---

# 2. `graph = build_graph()`：先构建工作流机器

这句：

```python
graph = build_graph()
```

会调用你的：

```python
src/research_agent/graph/workflow.py
```

里面大概是：

```python
def build_graph():
    workflow = StateGraph(AgentState)

    workflow.add_node("classify_task", classify_task)

    workflow.add_node("paper_node", paper_node)
    workflow.add_node("experiment_node", experiment_node)
    workflow.add_node("dataset_node", dataset_node)
    workflow.add_node("report_node", report_node)
    workflow.add_node("code_node", code_node)
    workflow.add_node("general_node", general_node)

    workflow.add_node("final_answer", final_answer_node)

    workflow.add_edge(START, "classify_task")

    workflow.add_conditional_edges(
        "classify_task",
        route_task,
        {
            "paper_node": "paper_node",
            "experiment_node": "experiment_node",
            "dataset_node": "dataset_node",
            "report_node": "report_node",
            "code_node": "code_node",
            "general_node": "general_node",
        },
    )

    workflow.add_edge("paper_node", "final_answer")
    workflow.add_edge("experiment_node", "final_answer")
    workflow.add_edge("dataset_node", "final_answer")
    workflow.add_edge("report_node", "final_answer")
    workflow.add_edge("code_node", "final_answer")
    workflow.add_edge("general_node", "final_answer")

    workflow.add_edge("final_answer", END)

    return workflow.compile()
```

它构建出来的是这张图：

```text
START
↓
classify_task
↓
route_task 条件路由
├── paper_node
├── experiment_node
├── dataset_node
├── report_node
├── code_node
└── general_node
↓
final_answer
↓
END
```

注意：

```python
graph = build_graph()
```

只是在“搭机器”，还没有处理任何用户问题。

真正开始处理问题的是：

```python
graph.invoke(...)
```

---

# 3. `create_initial_state(query)`：创建初始 State

测试脚本里每次循环都会执行：

```python
result = graph.invoke(create_initial_state(query))
```

其中 `create_initial_state(query)` 可能是：

```python
def create_initial_state(query: str) -> dict:
    return {
        "query": query,
        "task_type": "",
        "result": "",
        "final_answer": "",
        "classifier_source": "",
        "route_reason": "",
        "retrieved_docs": [],
        "sources": [],
    }
```

假设当前 query 是：

```text
OpenImages-MIAP 的性别标注是图像级还是 bbox 级？
```

那么初始 State 是：

```python
{
    "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
    "task_type": "",
    "result": "",
    "final_answer": "",
    "classifier_source": "",
    "route_reason": "",
    "retrieved_docs": [],
    "sources": [],
}
```

这就是 LangGraph 的“共享工作台”。

后续每个节点都会读取这个 State，并返回一部分字段更新它。

---

# 4. 第一步：进入 `classify_task`

图里第一条边是：

```python
workflow.add_edge(START, "classify_task")
```

所以 `graph.invoke()` 开始后，第一站一定是：

```text
classify_task
```

`classify_task` 的职责是：

> 根据用户输入判断任务类型，并把 `task_type`、`classifier_source`、`route_reason` 写回 State。

比如规则分类器可能判断：

```text
OpenImages-MIAP 的性别标注是图像级还是 bbox 级？
```

属于：

```python
task_type = "dataset_recommendation"
```

于是 `classify_task` 返回：

```python
{
    "task_type": "dataset_recommendation",
    "classifier_source": "rule",
    "route_reason": "命中了数据集 / OpenImages / MIAP 等相关关键词。"
}
```

LangGraph 会把这个返回值合并回原来的 State。

---

## classify_task 之后的 State

原来是：

```python
{
    "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
    "task_type": "",
    "result": "",
    "final_answer": "",
    "classifier_source": "",
    "route_reason": "",
    "retrieved_docs": [],
    "sources": [],
}
```

经过 `classify_task` 后变成：

```python
{
    "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
    "task_type": "dataset_recommendation",
    "result": "",
    "final_answer": "",
    "classifier_source": "rule",
    "route_reason": "命中了数据集 / OpenImages / MIAP 等相关关键词。",
    "retrieved_docs": [],
    "sources": [],
}
```

重点是：

```text
classify_task 没有直接调用 dataset_node
它只是写入 task_type
```

真正决定下一步去哪的是 `route_task`。

---

# 5. 第二步：执行 `route_task` 条件路由

你的图里有：

```python
workflow.add_conditional_edges(
    "classify_task",
    route_task,
    {
        "paper_node": "paper_node",
        "experiment_node": "experiment_node",
        "dataset_node": "dataset_node",
        "report_node": "report_node",
        "code_node": "code_node",
        "general_node": "general_node",
    },
)
```

意思是：

> `classify_task` 执行完后，不是固定去某个节点，而是调用 `route_task(state)` 决定下一站。

`route_task` 可能是：

```python
def route_task(state: AgentState):
    task_type = state["task_type"]

    if task_type == "paper_question":
        return "paper_node"
    elif task_type == "experiment_analysis":
        return "experiment_node"
    elif task_type == "dataset_recommendation":
        return "dataset_node"
    elif task_type == "report_generation":
        return "report_node"
    elif task_type == "code_help":
        return "code_node"
    else:
        return "general_node"
```

现在 State 里：

```python
state["task_type"] == "dataset_recommendation"
```

所以：

```python
route_task(state)
```

返回：

```python
"dataset_node"
```

于是 LangGraph 根据映射表进入：

```text
dataset_node
```

---

# 6. 第三步：进入 `dataset_node`

`dataset_node` 是 Day 9 改造后的 RAG 节点。

它大概是：

```python
def dataset_node(state: AgentState) -> dict:
    rag_result = run_rag_for_task(
        state=state,
        task_type="dataset_recommendation",
        top_k=3,
    )

    result = f"""
这是数据集推荐 / 数据集说明任务。系统已从数据集资料库中检索到相关内容。

基于检索资料的初步回答：
{build_simple_answer_from_context(state["query"], rag_result["retrieved_context"])}
""".strip()

    return {
        "retrieved_docs": rag_result["retrieved_docs"],
        "sources": rag_result["sources"],
        "result": result,
    }
```

这个节点做了三件事：

```text
1. 读取 state["query"]
2. 调用 Retriever 检索 dataset_doc
3. 把 retrieved_docs、sources、result 写回 State
```

---

# 7. `run_rag_for_task()` 内部发生了什么？

`dataset_node` 里调用：

```python
rag_result = run_rag_for_task(
    state=state,
    task_type="dataset_recommendation",
    top_k=3,
)
```

这个函数大概是：

```python
def run_rag_for_task(state: AgentState, task_type: str, top_k: int = 3) -> dict:
    query = state["query"]

    docs = retrieve_documents(
        query=query,
        task_type=task_type,
        top_k=top_k,
        use_filter=True,
    )

    retrieved_docs = [document_to_dict(doc) for doc in docs]
    sources = extract_sources_from_docs(docs)
    retrieved_context = format_retrieved_docs(docs, max_chars_per_doc=500)

    return {
        "retrieved_docs": retrieved_docs,
        "sources": sources,
        "retrieved_context": retrieved_context,
    }
```

假设 query 是：

```text
OpenImages-MIAP 的性别标注是图像级还是 bbox 级？
```

task_type 是：

```python
"dataset_recommendation"
```

那么 `retrieve_documents()` 会做：

```python
metadata_filter = {"source_type": "dataset_doc"}
```

然后从 Chroma 里只检索：

```text
dataset_doc 类型的 chunks
```

所以它不会查论文笔记，也不会查实验文档。

---

# 8. Retriever 返回什么？

`retrieve_documents()` 返回的是一个 `Document` 列表，比如：

```python
docs = [
    Document(
        page_content="OpenImages-MIAP 是基于 OpenImages 的人物属性相关标注数据……",
        metadata={
            "source_type": "dataset_doc",
            "dataset": "OpenImages-MIAP",
            "path": "data/datasets/openimages_miap.md",
            "chunk_id": 12,
        },
    ),
    Document(
        page_content="该数据集的关键特点是属性更接近 person-level 或 bbox-level，而不是简单 image-level 标签……",
        metadata={
            "source_type": "dataset_doc",
            "dataset": "OpenImages-MIAP",
            "path": "data/datasets/openimages_miap.md",
            "chunk_id": 13,
        },
    ),
    Document(
        page_content="在做图像级偏见评估时，需要特别注意一张图中多个人物属性并存的问题……",
        metadata={
            "source_type": "dataset_doc",
            "dataset": "OpenImages-MIAP",
            "path": "data/datasets/openimages_miap.md",
            "chunk_id": 14,
        },
    ),
]
```

注意：这里返回的是 **chunk 级 Document**，不是整篇文件。

---

# 9. `document_to_dict()`：把 Document 转成 State 里更好保存的 dict

因为 `Document` 是对象，放进 State 里不是不行，但不如普通 dict 好打印、好保存、好调试。

所以：

```python
retrieved_docs = [document_to_dict(doc) for doc in docs]
```

会把每个 `Document` 变成：

```python
{
    "content": "OpenImages-MIAP 是基于 OpenImages 的人物属性相关标注数据……",
    "metadata": {
        "source_type": "dataset_doc",
        "dataset": "OpenImages-MIAP",
        "path": "data/datasets/openimages_miap.md",
        "chunk_id": 12,
    }
}
```

所以 `retrieved_docs` 大概是：

```python
[
    {
        "content": "...chunk 1 正文...",
        "metadata": {"path": "data/datasets/openimages_miap.md", ...}
    },
    {
        "content": "...chunk 2 正文...",
        "metadata": {"path": "data/datasets/openimages_miap.md", ...}
    },
    {
        "content": "...chunk 3 正文...",
        "metadata": {"path": "data/datasets/openimages_miap.md", ...}
    },
]
```

---

# 10. `extract_sources_from_docs()`：提取 Sources，并按文件去重

如果 3 个 chunk 都来自：

```text
data/datasets/openimages_miap.md
```

那么 `docs` 有 3 个，但 Sources 只显示 1 个文件。

```python
sources = extract_sources_from_docs(docs)
```

结果可能是：

```python
[
    {
        "path": "data/datasets/openimages_miap.md",
        "source_type": "dataset_doc",
        "title": "OpenImages-MIAP Dataset",
        "dataset": "OpenImages-MIAP",
        "run_tag": "",
    }
]
```

也就是：

```text
retrieved_docs：保留 chunk 级信息
sources：按文件 path 去重
```

这就是为什么你测试脚本最后可能看到：

```text
retrieved_docs count: 3
sources count: 1
```

这是正常的。

---

# 11. `format_retrieved_docs()`：把检索结果整理成字符串

这一步是为了生成当前 v0.2 的简单回答。

```python
retrieved_context = format_retrieved_docs(docs, max_chars_per_doc=500)
```

可能得到：

```text
[1] source_type=dataset_doc | path=data/datasets/openimages_miap.md | dataset=OpenImages-MIAP
OpenImages-MIAP 是基于 OpenImages 的人物属性相关标注数据……

[2] source_type=dataset_doc | path=data/datasets/openimages_miap.md | dataset=OpenImages-MIAP
该数据集的关键特点是属性更接近 person-level 或 bbox-level……

[3] source_type=dataset_doc | path=data/datasets/openimages_miap.md | dataset=OpenImages-MIAP
在做图像级偏见评估时，需要特别注意……
```

然后 `build_simple_answer_from_context()` 会把这些资料拼进回答里。

---

# 12. `dataset_node` 返回什么？

最终 `dataset_node` 返回：

```python
{
    "retrieved_docs": [
        {
            "content": "...chunk 1...",
            "metadata": {...}
        },
        {
            "content": "...chunk 2...",
            "metadata": {...}
        },
        {
            "content": "...chunk 3...",
            "metadata": {...}
        },
    ],
    "sources": [
        {
            "path": "data/datasets/openimages_miap.md",
            "source_type": "dataset_doc",
            "title": "OpenImages-MIAP Dataset",
            "dataset": "OpenImages-MIAP",
            "run_tag": "",
        }
    ],
    "result": "这是数据集推荐 / 数据集说明任务。系统已从数据集资料库中检索到相关内容。\n\n基于检索资料的初步回答：..."
}
```

LangGraph 会把这些字段合并回 State。

---

## dataset_node 之后的 State

这时 State 从：

```python
{
    "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
    "task_type": "dataset_recommendation",
    "result": "",
    "final_answer": "",
    "classifier_source": "rule",
    "route_reason": "...",
    "retrieved_docs": [],
    "sources": [],
}
```

变成：

```python
{
    "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
    "task_type": "dataset_recommendation",
    "result": "这是数据集推荐 / 数据集说明任务。系统已从数据集资料库中检索到相关内容。\n\n基于检索资料的初步回答：...",
    "final_answer": "",
    "classifier_source": "rule",
    "route_reason": "...",
    "retrieved_docs": [
        {"content": "...", "metadata": {...}},
        {"content": "...", "metadata": {...}},
        {"content": "...", "metadata": {...}},
    ],
    "sources": [
        {
            "path": "data/datasets/openimages_miap.md",
            "source_type": "dataset_doc",
            "dataset": "OpenImages-MIAP",
        }
    ],
}
```

---

# 13. 第四步：固定边进入 `final_answer`

你的 workflow 里有：

```python
workflow.add_edge("dataset_node", "final_answer")
```

所以 `dataset_node` 执行完后，一定进入：

```text
final_answer
```

也就是调用：

```python
final_answer_node(state)
```

---

# 14. `final_answer_node` 怎么生成最终输出？

`final_answer_node` 大概是：

```python
def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
分类来源：{state["classifier_source"]}
分类原因：{state["route_reason"]}

回答：
{state["result"]}

Sources:
{format_sources(state.get("sources", []))}
""".strip()

    return {
        "final_answer": final_answer
    }
```

它读取 State 里的这些字段：

```python
state["task_type"]
state["classifier_source"]
state["route_reason"]
state["result"]
state["sources"]
```

然后生成一个最终字符串。

比如：

```text
任务类型：dataset_recommendation
分类来源：rule
分类原因：命中了数据集 / OpenImages / MIAP 等相关关键词。

回答：
这是数据集推荐 / 数据集说明任务。系统已从数据集资料库中检索到相关内容。

基于检索资料的初步回答：
用户问题：OpenImages-MIAP 的性别标注是图像级还是 bbox 级？

检索到的相关资料摘要如下：
[1] source_type=dataset_doc | path=data/datasets/openimages_miap.md | dataset=OpenImages-MIAP
OpenImages-MIAP 是基于 OpenImages 的人物属性相关标注数据……

Sources:
1. [dataset_doc] data/datasets/openimages_miap.md (dataset=OpenImages-MIAP)
```

`final_answer_node` 返回：

```python
{
    "final_answer": "任务类型：dataset_recommendation\n分类来源：rule\n..."
}
```

LangGraph 再次合并回 State。

---

# 15. 最终 State 长什么样？

`final_answer_node` 之后，State 里所有关键字段都齐了：

```python
{
    "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",

    "task_type": "dataset_recommendation",
    "classifier_source": "rule",
    "route_reason": "命中了数据集 / OpenImages / MIAP 等相关关键词。",

    "retrieved_docs": [
        {"content": "...chunk 1...", "metadata": {...}},
        {"content": "...chunk 2...", "metadata": {...}},
        {"content": "...chunk 3...", "metadata": {...}},
    ],

    "sources": [
        {
            "path": "data/datasets/openimages_miap.md",
            "source_type": "dataset_doc",
            "dataset": "OpenImages-MIAP",
        }
    ],

    "result": "这是数据集推荐 / 数据集说明任务。系统已从数据集资料库中检索到相关内容。\n...",

    "final_answer": "任务类型：dataset_recommendation\n分类来源：rule\n分类原因：...\n\n回答：...\n\nSources:\n1. [dataset_doc] data/datasets/openimages_miap.md ..."
}
```

然后 graph 到达：

```text
END
```

`graph.invoke()` 返回这个完整 State。

---

# 16. 测试脚本为什么可以打印这些？

测试脚本里：

```python
result = graph.invoke(create_initial_state(query))
```

这里的 `result` 就是最终 State。

所以：

```python
print(result["final_answer"])
```

打印最终回答。

```python
print("\nretrieved_docs count:", len(result.get("retrieved_docs", [])))
```

打印检索到的 chunk 数量。

如果 `top_k=3`，通常是：

```text
retrieved_docs count: 3
```

但如果没有检索到资料，可能是 0。

```python
print("sources count:", len(result.get("sources", [])))
```

打印去重后的来源文件数量。

如果 3 个 chunk 都来自同一个文件，可能是：

```text
sources count: 1
```

这就是你前面问的那个问题。

---

# 17. 换一个实验问题，State 怎么变？

假设 query 是：

```text
请帮我解释 coco_val_n300_g1 这个实验的目的
```

初始 State：

```python
{
    "query": "请帮我解释 coco_val_n300_g1 这个实验的目的",
    "task_type": "",
    "result": "",
    "final_answer": "",
    "classifier_source": "",
    "route_reason": "",
    "retrieved_docs": [],
    "sources": [],
}
```

`classify_task` 后：

```python
{
    "query": "请帮我解释 coco_val_n300_g1 这个实验的目的",
    "task_type": "experiment_analysis",
    "classifier_source": "rule",
    "route_reason": "命中了实验 / run_tag / coco_val_n300_g1 相关关键词。",
    ...
}
```

`route_task` 返回：

```python
"experiment_node"
```

进入 `experiment_node`。

`experiment_node` 调用：

```python
retrieve_documents(
    query="请帮我解释 coco_val_n300_g1 这个实验的目的",
    task_type="experiment_analysis",
    top_k=3,
    use_filter=True,
)
```

Day 8 的映射：

```python
"experiment_analysis" → "experiment_doc"
```

所以 Chroma filter 是：

```python
{"source_type": "experiment_doc"}
```

返回结果主要来自：

```text
data/experiments/coco_val_n300_g1.md
```

于是最终 State 里的 sources 可能是：

```python
[
    {
        "path": "data/experiments/coco_val_n300_g1.md",
        "source_type": "experiment_doc",
        "run_tag": "coco_val_n300_g1",
        "dataset": "COCO2017",
    }
]
```

最终回答里 Sources 显示：

```text
1. [experiment_doc] data/experiments/coco_val_n300_g1.md (dataset=COCO2017, run_tag=coco_val_n300_g1)
```

---

# 18. 再换一个代码报错问题，流程有什么不同？

query：

```text
ModuleNotFoundError: No module named langgraph 怎么解决
```

`classify_task` 可能得到：

```python
task_type = "code_help"
```

`route_task` 返回：

```python
"code_node"
```

但 Day 9 里 `code_node` 还没有接 RAG。

所以 `code_node` 返回：

```python
{
    "retrieved_docs": [],
    "sources": [],
    "result": "这是代码辅助任务，后续会接入代码解释与修改工具。"
}
```

之后 `final_answer_node` 生成：

```text
任务类型：code_help
分类来源：rule
分类原因：命中了代码、报错或环境相关关键词。

回答：
这是代码辅助任务，后续会接入代码解释与修改工具。

Sources:
无
```

测试脚本会打印：

```text
retrieved_docs count: 0
sources count: 0
```

这也是正常的。

因为 Day 9 只把 RAG 接入了：

```text
paper_node
experiment_node
dataset_node
```

还没接入：

```text
code_node
report_node
general_node
```

---

# 19. 整个 LangGraph 的 State 变化总览

以 dataset 问题为例，可以画成这样：

```text
初始 State
{
  query: "OpenImages-MIAP ...",
  task_type: "",
  result: "",
  final_answer: "",
  classifier_source: "",
  route_reason: "",
  retrieved_docs: [],
  sources: []
}

↓ classify_task

{
  query: "OpenImages-MIAP ...",
  task_type: "dataset_recommendation",
  classifier_source: "rule",
  route_reason: "命中数据集关键词",
  result: "",
  final_answer: "",
  retrieved_docs: [],
  sources: []
}

↓ route_task(state) 返回 "dataset_node"

↓ dataset_node

{
  query: "OpenImages-MIAP ...",
  task_type: "dataset_recommendation",
  classifier_source: "rule",
  route_reason: "命中数据集关键词",
  retrieved_docs: [chunk1, chunk2, chunk3],
  sources: [openimages_miap.md],
  result: "这是数据集推荐任务...检索到相关内容...",
  final_answer: ""
}

↓ final_answer_node

{
  query: "OpenImages-MIAP ...",
  task_type: "dataset_recommendation",
  classifier_source: "rule",
  route_reason: "命中数据集关键词",
  retrieved_docs: [chunk1, chunk2, chunk3],
  sources: [openimages_miap.md],
  result: "这是数据集推荐任务...检索到相关内容...",
  final_answer: "任务类型：dataset_recommendation\n分类来源：rule\n..."
}

↓ END

graph.invoke 返回最终 State
```

---

# 20. 最关键的理解

你可以把 Day 9 的 LangGraph 理解成：

```text
classify_task：
负责写 task_type

route_task：
负责看 task_type，决定去哪个任务节点

paper / experiment / dataset node：
负责调用 Retriever，写 retrieved_docs、sources、result

final_answer_node：
负责读取 State 里的所有结果，生成 final_answer
```

也就是：

```text
分类节点写“任务身份”
路由函数选“处理路径”
任务节点写“检索结果和回答草稿”
最终节点写“用户看到的最终文本”
```

---

# 21. 为什么这已经是 Agentic RAG？

因为检索不是一上来就查全部，而是由 LangGraph 的状态决定：

```text
query
↓
classify_task 写 task_type
↓
route_task 选择节点
↓
节点根据 task_type 调用 retrieve_documents
↓
retrieve_documents 根据 task_type 设置 metadata filter
```

所以它不是普通 RAG：

```text
query → retrieve all → answer
```

而是：

```text
query → classify → route → task-specific retrieve → answer with sources
```

这就是阶段二最核心的成果。
